# Report-based pursuit and avoidance match

Run all cells for a 10-second match. Each second spent 0.3–0.6 m behind the opponent within ±15° earns one point. Both drones hold 0.5 m altitude and constant yaw. Diamonds mark avoidance. Symmetric starts can produce a zero-score draw.

The shared planar search uses the existing ellipsoidal VO geometry, predicts two seconds ahead, and checks command speed, acceleration, bounds and separation. A right-side passing preference is a tuning assumption. The second-order plant is an uncalibrated simulation; this does not validate hardware flight.


In [1]:
# Locate the project even when the kernel starts at the workspace root.
from pathlib import Path
import sys
import importlib
candidates = [Path.cwd(), Path.cwd() / 'Project-mocap-pull',
              Path.cwd() / 'crazyflie-client' / 'Project-mocap-pull',
              Path.cwd() / 'AE483-Pursuit-Drone' / 'crazyflie-client' / 'Project-mocap-pull']
project = next((p for p in candidates if (p / 'match_simulation.py').exists()), None)
if project is None:
    raise RuntimeError('Open this notebook from the ae483 workspace or Project-mocap-pull folder.')
sys.path.insert(0, str(project.resolve()))
for module in ('match_config', 'tracking', 'pursuit_avoidance', 'match_simulation'):
    if module in sys.modules:
        importlib.reload(sys.modules[module])

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Rectangle
from IPython.display import HTML, display
from match_config import CONFIG
from match_simulation import simulate_match, validate_match

# Shared report limits; change match_config.py to tune flight and simulations.
# "head_on" is symmetric; "rear_start" demonstrates nonzero scoring.
scenario = "head_on"
result = (simulate_match() if scenario == "head_on" else
          simulate_match(starts=[[-0.5, 0, CONFIG.height], [0, 0, CONFIG.height]],
                         initial_headings=[[1, 0], [1, 0]]))
summary = validate_match(result)
print(summary)
period = CONFIG.period
box_min, box_max = CONFIG.lower, CONFIG.upper
times = result['time']
paths = result['positions'][..., :2]
targets = result['targets']
headings = result['headings']
modes = result['statuses']
scores = result['scores']
outcome = result['outcome']



{'duration': 10.0, 'min_q': 1.5889316455349034, 'max_command_speed': 0.6, 'max_command_acceleration': 1.5000000000000009, 'scores': [0.0, 0.0], 'outcome': 'Draw'}


In [2]:
fig, ax = plt.subplots(figsize=(7, 7), dpi=110)
ax.add_patch(Rectangle(box_min[:2], *(box_max[:2] - box_min[:2]),
                       fill=False, linestyle="--", color="gray"))
ax.set(xlim=(-2.2, 2.2), ylim=(-2.2, 2.2), xlabel="World x (m)",
       ylabel="World y (m)", title="Opponent 1 vs opponent 2")
ax.set_aspect("equal")
ax.grid(alpha=0.2)
colors = ["tab:blue", "tab:orange"]
dots, trails, stars, links, directions = [], [], [], [], []
for i, color in enumerate(colors):
    dots.append(ax.plot([], [], "o", color=color, markersize=9,
                        label=f"Opponent {i+1}")[0])
    trails.append(ax.plot([], [], color=color, alpha=0.3)[0])
    stars.append(ax.plot([], [], "*", color=color, markersize=12,
                         label=f"Target for opponent {i+1}")[0])
    links.append(ax.plot([], [], ":", color=color, alpha=0.5)[0])
    directions.append(ax.plot([], [], color=color, linewidth=2)[0])
status = ax.text(0.02, 0.98, "", transform=ax.transAxes, fontsize=9, va="top",
                 bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"))
ax.legend(loc="lower left", fontsize=8)


def update(frame):
    for i in range(2):
        position = paths[frame, i]
        target = targets[frame, i]
        other = paths[frame, 1-i]
        dots[i].set_data([position[0]], [position[1]])
        dots[i].set_marker("D" if modes[frame][i] not in ("tracking", "free") else "o")
        trails[i].set_data(paths[:frame+1, i, 0], paths[:frame+1, i, 1])
        stars[i].set_data([target[0]], [target[1]])
        links[i].set_data([other[0], target[0]], [other[1], target[1]])
        forward = position + 0.12 * headings[frame, i]
        directions[i].set_data([position[0], forward[0]], [position[1], forward[1]])
    label = outcome if frame == len(times)-1 else " / ".join(modes[frame])
    label += f" | score {scores[frame, 0]:.1f} : {scores[frame, 1]:.1f}"
    status.set_text(f"{times[frame]:.1f} s | {label}")
    return dots + trails + stars + links + directions + [status]


# Hold the final frame briefly so the outcome is easy to read.
playback_stride = 1  # Smooth 20 fps playback, matching the simulation updates.
frames = list(range(0, len(times), playback_stride))
if frames[-1] != len(times)-1:
    frames.append(len(times)-1)
frames += [len(times)-1] * 10
animation = FuncAnimation(fig, update, frames=frames,
                          interval=period * playback_stride * 1000, repeat=False)
plt.close(fig)
html = animation.to_jshtml()
output_dir = project / 'simulation_output'
output_dir.mkdir(exist_ok=True)
(output_dir / 'match_simulation.html').write_text(html)
print(f"Saved playback: {output_dir / 'match_simulation.html'}")
display(HTML(html))


Saved playback: /Users/anirudhvishneek/ae483/AE483-Pursuit-Drone/crazyflie-client/Project-mocap-pull/simulation_output/match_simulation.html
